## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [11]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [1]:
!ollma list

'ollma' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
#MODEL = "gpt-4.1-nano"
MODEL = "gemma4:e2b"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [4]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [12]:
retriever = vectorstore.as_retriever()
#llm = ChatOpenAI(temperature=0, model_name=MODEL)
llm = ChatOllama(temperature=0, model=MODEL)

### These LangChain objects implement the method `invoke()`

In [6]:
retriever.invoke("Who is Avery?")

[Document(id='2f173e2a-95f8-4773-9127-4b19a2c9efae', metadata={'source': 'knowledge-base\\employees\\Avery Lancaster.md', 'doc_type': 'employees'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and a

In [13]:
llm.invoke("Who is Avery?")

AIMessage(content='Please provide more context. "Avery" is a common name, and I need more information to tell you who you are referring to.\n\nAre you asking about:\n\n*   A specific person (an actor, historical figure, etc.)?\n*   A character in a book, movie, or TV show?\n*   A public figure?\n\nIf you can give me more details, I would be happy to help!', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-10-15T13:22:40.1583583Z', 'done': True, 'done_reason': 'stop', 'total_duration': 155147055200, 'load_duration': 133809827600, 'prompt_eval_count': 20, 'prompt_eval_duration': 610560000, 'eval_count': 295, 'eval_duration': 20651276000, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--8e601d98-96c4-4b9a-a1cf-481453e81567-0', usage_metadata={'input_tokens': 20, 'output_tokens': 295, 'total_tokens': 315})

## Time to put this together!

In [9]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [14]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [15]:
answer_question("Who is Averi Lancaster?", [])

"Avery Lancaster is a key figure at Insurellm. Here is a summary of her profile:\n\n*   **Job Title:** Co-Founder & Chief Executive Officer (CEO)\n*   **Location:** San Francisco, California\n*   **Role at Insurellm:** She co-founded Insurellm in 2015 and has guided the company to become a leading Insurance Tech provider, known for her innovative leadership strategies and risk management expertise.\n*   **Career History:** Before founding Insurellm, she was a Senior Product Manager at Innovate Insurance Solutions.\n*   **Recent Role:** Since January 2021, she has served as a Senior Data Engineer, where she has successfully led projects that have improved data retrieval times and has been recognized as the Insurellm Innovator of the year in 2023.\n\nShe is known for her innovative leadership and ability to drive the company's growth."

## What could possibly come next? 😂

In [16]:
gr.ChatInterface(answer_question).launch()

c:\Users\Chelsea_amrit\LLM\llm start\llm_engineering\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!